## Task2 学习总结（第二章：核心概念）

### 学到了什么
- state 是"本次判断允许看的证据"，设计时要防答案泄漏
- Noul/Choice/Score 是答案空间的三种形状：是不是 / 选哪个 / 程度多少
- 不能只看 argmax：[0.51, 0.49] 和 [0.99, 0.01] 选出同一个答案，但不确定性完全不同
- 模型做判断，代码做决策：阈值、分流、逃生舱都写在 decide() 里

### 遇到的问题
- （写你实际遇到的问题，比如哪个概念一开始没绕过来）

### 一点感想
- （一句话即可）

### 课程出处
- 教程：https://github.com/datawhalechina/jev-cookbook

In [1]:
ticket = {
    "ticket_id": "YG-2026-10132",
    "message": "我上周买的蓝牙耳机到货后发现右耳没有声音，申请换货，已经把问题视频上传了。",
    "order_status": "已签收3天",
    "member_tier": "金牌会员",
    "amount": 299,
    "category": "数码配件",
}

In [2]:
noul_need_human = 0.15
print(f"Noul 是否需要转人工: P(true) = {noul_need_human}")
choice_queues = {"退款队列": 0.08, "换货队列": 0.87, "咨询队列": 0.05}
print(f"Choice 队列分布: {choice_queues}")
score_urgency = {1: 0.02, 2: 0.08, 3: 0.55, 4: 0.30, 5: 0.05}
print(f"Score 紧急程度分布: {score_urgency}")

print("\n同一张工单，三种问法，三种答案形状：")
print("- Noul 答'是不是'，Choice 答'选哪个'，Score 答'程度多少'")

Noul 是否需要转人工: P(true) = 0.15
Choice 队列分布: {'退款队列': 0.08, '换货队列': 0.87, '咨询队列': 0.05}
Score 紧急程度分布: {1: 0.02, 2: 0.08, 3: 0.55, 4: 0.3, 5: 0.05}

同一张工单，三种问法，三种答案形状：
- Noul 答'是不是'，Choice 答'选哪个'，Score 答'程度多少'


In [3]:
case_a = {"退款队列": 0.51, "换货队列": 0.49, "咨询队列": 0.00}  # 纠结
case_b = {"退款队列": 0.01, "换货队列": 0.99, "咨询队列": 0.00}  # 明确

def argmax(d):
    return max(d, key=d.get)

print(f"Case A argmax: {argmax(case_a)}")
print(f"Case B argmax: {argmax(case_b)}")
print("\nargmax 都是'换货队列'，但：")
print("- Case A [0.51, 0.49]：模型在退款/换货之间纠结 → 不能直接执行")
print("- Case B [0.99, 0.01]：模型高度确定 → 可以直接分流")
print("\n结论：只看'选了谁'会丢掉不确定性信息，分布形状才是决策依据。")
print("这就是第二章的核心：把概率分布用起来，而不是只取 argmax。")

Case A argmax: 退款队列
Case B argmax: 换货队列

argmax 都是'换货队列'，但：
- Case A [0.51, 0.49]：模型在退款/换货之间纠结 → 不能直接执行
- Case B [0.99, 0.01]：模型高度确定 → 可以直接分流

结论：只看'选了谁'会丢掉不确定性信息，分布形状才是决策依据。
这就是第二章的核心：把概率分布用起来，而不是只取 argmax。


In [4]:
def decide(choice_probs, noul_prob):
    labels = list(choice_probs.keys())
    probs = list(choice_probs.values())
    top_idx = probs.index(max(probs))
    top_label, top_prob = labels[top_idx], probs[top_idx]

    # 规则1：Noul 逃生舱 —— 先检查，不满足直接转人工
    if noul_prob > 0.7:
        return f"转人工（noul={noul_prob:.2f} 过高）"
    # 规则2：置信度检查 —— 分布太平说明模型不确定
    if top_prob < 0.6:
        return f"转人工复核（最高概率仅 {top_prob:.2f}）"
    # 规则3：正常分流
    return f"分配到：{top_label}（置信度 {top_prob:.2f}）"

In [1]:
print("正常工单:", decide(choice_queues, noul_need_human))
print("纠结工单:", decide(case_a, 0.10))
print("高危工单:", decide(case_b, 0.85))
vague = {"退款队列": 0.34, "换货队列": 0.33, "咨询队列": 0.33}
print("含糊工单:", decide(vague, 0.20))

NameError: name 'decide' is not defined